# MicroDuck · 앉아서 두리번거리기와 인사

목표: **앉은 상태 유지 → 왼쪽 → 오른쪽 → 정면 → 두 번 끄덕이기 → 앉아서 정지**.
좌우 약 15°, 끄덕임 약 8°의 작은 명령을 14초 동안 부드럽게 줍니다.

**오늘은 실행하지 않아도 됩니다.** 내일 A100 GPU를 선택하고 아래 셀을 순서대로 실행하세요.
기본 `START_TRAINING=False`이므로 전체 실행만으로 학습이 시작되지 않습니다.

JumpSpin 모델은 사용하지 않습니다. 새 실험은 `seated_greeting_v1`이며 자세 검증 실패 시 본학습을 중단합니다.
이 정책은 이미 앉은 상태에서만 사용하도록 설계됐으며, 실제 로봇 배포용 검증은 별도입니다.

[상세 가이드](https://github.com/namyikim/microduck/blob/main/doc/seated-greeting.md)


In [ ]:
START_TRAINING = False  # 내일 직접 True로 바꾼 후 학습 셀 실행
TARGET_ITERS = 3000
NUM_ENVS = 4096
REPO_DIR = "/content/microduck"
DRIVE_ROOT = "/content/drive/MyDrive/microduck-training/seated_greeting_v1"


## 1. Drive 연결 / Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. 설치 / Install
런타임 유형에서 **A100 GPU**를 선택하세요.

In [ ]:
import os, subprocess
from pathlib import Path
if not Path(REPO_DIR, '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/namyikim/microduck.git', REPO_DIR], check=True)
subprocess.run(['bash', f'{REPO_DIR}/scripts/setup_colab.sh'], env=dict(os.environ, REPO_DIR=REPO_DIR), check=True)
os.environ['PATH'] = str(Path.home()/'.local/bin') + ':' + os.environ['PATH']


## 3. 학습 시작 / Resume

`START_TRAINING=True`일 때만 실행합니다.

1. 64환경에서 앉은 자세 3초 유지 + 머리 동작 물리 검증
2. 64환경·5회 GPU 검증 학습 + 공식 ONNX 내보내기·61→14 입력/출력 검증
3. 4,096환경 본학습 — 같은 실험의 최신 체크포인트 자동 재개
4. 최종 ONNX와 평가 영상 5개, 접촉·기울기·동작 추적 결과 저장

3,000회는 초기 예산이며 동작 성공을 보장하는 횟수가 아닙니다. 실패 결과도 영상과 JSON에 남습니다.


In [ ]:
if not START_TRAINING:
    print('학습 대기 상태입니다. 내일 설정 셀에서 START_TRAINING=True로 바꾸세요.')
else:
    subprocess.run([
        'uv', 'run', 'python', 'scripts/train_seated_greeting_colab.py',
        '--start-training', '--target-iters', str(TARGET_ITERS),
        '--num-envs', str(NUM_ENVS), '--drive-root', DRIVE_ROOT,
    ], cwd=REPO_DIR, check=True)


## 4. 결과 확인 / Results

Drive → `microduck-training/seated_greeting_v1/`

- `training.log`: 본학습 로그
- `logs/rsl_rl/seated_greeting_v1/`: 모델
- `evaluations/<실행시각>/`: `preflight.json`, `policy.onnx`, 영상 5개, `evaluation_summary.json`, 시계열 JSON
- `status.json`: 단계 또는 실패 원인

머리 접촉·20° 초과 기울기·앉은 높이 이탈·좌우 보기나 끄덕임 미수행을 검사합니다.
`sim_checks_passed`는 시뮬레이션 검사 결과이며 하드웨어 안전 인증이 아닙니다.

## English

Train a **seated greeting**: look left, look right, face forward, nod twice, and remain seated.
This notebook starts with training **disabled**. On a later day, select an A100 GPU, mount Drive,
install the project, set `START_TRAINING=True`, and run the training cell.
The pipeline requires a seated-pose/head-motion preflight, a 64-environment / 5-iteration smoke
run and official normalized ONNX validation before the main run. It resumes only this experiment,
then saves five videos and diagnostics. The 3,000-iteration budget does not guarantee success.
The model neither teaches sitting down nor replaces the standing/walking controller.
